In [1]:
import pandas as pd

BASE = "../data/"

# Carregamento de dados
df_vendas = pd.read_csv(BASE + 'vendas.csv')
df_mkt = pd.read_csv(BASE + 'marketing.csv')
df_clientes = pd.read_csv(BASE + 'clientes.csv')
df_atendimento = pd.read_csv(BASE + 'atendimento.csv')
df_estoque = pd.read_csv(BASE + 'estoque.csv')

# Profiling
nulos_vendas = df_vendas.isnull().sum().max() 
margem_negativa = (df_vendas['margem_contribuicao'] < 0).sum()

print(f"Nulos máximos em Vendas/Atendimento: {nulos_vendas}")
print(f"Pedidos com margem negativa: {margem_negativa}")

# Tratamento: Remoção da única linha nula marginal (sem impacto estatístico)
df_vendas.dropna(inplace=True)
df_atendimento.dropna(inplace=True)

Nulos máximos em Vendas/Atendimento: 1
Pedidos com margem negativa: 491


In [ ]:
# Abertura de Margem por Canal
exploracao_canal = df_vendas.groupby('canal').agg(
    receita=('receita_liquida', 'sum'),
    margem=('margem_contribuicao', 'sum')
)
exploracao_canal['margem_pct'] = exploracao_canal['margem'] / exploracao_canal['receita']
print(exploracao_canal.sort_values('margem_pct'))

# Devoluções
taxa_devolucao = df_vendas['devolvido'].mean()
print(f"Taxa de devolução geral: {taxa_devolucao:.2%}")

In [8]:
# H1: Margem 
# Validado na etapa 2 (Marketplace drena margem).

# H2: Qualidade da Aquisição
mkt_roi = df_mkt.groupby('canal').agg(
    investimento=('investimento_reais', 'sum'),
    conversoes=('conversoes', 'sum'),
    receita=('receita_gerada', 'sum')
)
mkt_roi['cac'] = mkt_roi['investimento'] / mkt_roi['conversoes']
mkt_roi['roas'] = mkt_roi['receita'] / mkt_roi['investimento']

print(mkt_roi)

# H3: Falhas Operacionais Pós-Venda
custo_devolucao = df_vendas[df_vendas['devolvido'] == True]['receita_liquida'].sum()

print("\n")
print(custo_devolucao)

# H4: Causa-Raiz no Atendimento
cs_vol = df_atendimento.groupby('categoria_problema').agg(
    volume=('ticket_id', 'count'), custo=('custo_operacional_ticket', 'sum')
).sort_values('volume', ascending=False)

print("\n")
print(cs_vol)

# H5: Segmentos de Clientes
ltv_seg = df_clientes.groupby('segmento_rfm')['ltv_acumulado'].mean().sort_values(ascending=False)

print("\n")
print(ltv_seg)

                 investimento  conversoes       receita       cac      roas
canal                                                                      
Email Marketing   29700531.98    16563197  9.104621e+07  1.793164  3.065474
Google Ads        30224839.25    16118323  1.065574e+08  1.875185  3.525491
Influenciador     28061599.02    15940484  2.175780e+08  1.760398  7.753585
Instagram Ads     30165772.33    15360484  1.359386e+08  1.963856  4.506385
Marketplace       32682629.69    17894187  9.879591e+07  1.826438  3.022888
Orgânico          29808867.75    16401820  8.964926e+07  1.817412  3.007469
TikTok Ads        29770101.07    17009948  1.390634e+08  1.750158  4.671243


2816701.5500000003


                        volume     custo
categoria_problema                      
Onde está meu pedido?    10765  159660.0
Defeito                   6509   96592.0
Troca de Tamanho          5360   79325.0
Dúvida Técnica            5314   78888.0
Pagamento não aprovado    4281   63856.0
Elogio

In [ ]:
# Premissas (Base x Alavanca x Captura)

# Oportunidade 1: Automação de CS
# Base: Custo dos tickets de rastreio (R$ 159.660)
# Alavanca: Implementação de Bot de IA (redução de volume)
# Taxa de Captura: 80% 
ganho_cs = 159660 * 0.80 # R$ 127.728

# Oportunidade 2: Mitigação de Devoluções
# Base: Receita perdida (R$ 2.816.701)
# Alavanca: Margem média salva (54.2%)
# Taxa de Captura: 20% de redução na taxa de devolução
ganho_dev = (2816701 * 0.542) * 0.20 # R$ 305.330

# Oportunidade 3: Ajuste Marketplace
# Base: Receita do MKTPlace (R$ 3.986.568)
# Alavanca: Aumento de margem de 51.4% para 55.0% (delta 3.6%)
# Taxa de Captura: 100% da adequação
ganho_mktplace = 3986568 * 0.036 # R$ 143.516

### Tabela 1: Diagnóstico de Rentabilidade e Operações Comerciais

| Item | Descrição |
| :--- | :--- |
| **Fato** | Taxa de devolução geral em 14,8% e 491 pedidos faturados com margem de contribuição negativa. |
| **Causa** | Falhas operacionais de especificação de produto e ausência de travas de *pricing*/frete dinâmico no canal Marketplace (margem média de 51,4% vs. ~55% nos demais canais). |
| **Implicação** | Perda anual estimada de R$ 305.330 em margem operacional por devoluções excessivas e R$ 143.516 por ineficiência de margem no Marketplace (~R$ 450 mil/ano combinados). |
| **Ação Recomendada** | Travar sistemicamente pedidos com margem unitária negativa; Auditar fichas técnicas/tabelas de medidas dos top SKUs devolvidos;  Reprecificar repasse de frete no Marketplace.   


---

### Tabela 2: Diagnóstico de Atendimento e Eficiência Operacional

| Item | Descrição |
| :--- | :--- |
| **Fato** | 10.765 tickets de suporte abertos exclusivamente sob a categoria "Onde está meu pedido?". |
| **Causa** | Falta de rastreamento logístico proativo e ausência de integração transacional entre a transportadora/ERP e os canais de contato com o cliente. |
| **Implicação** | R$ 159.660 consumidos em custo direto de suporte humano N1, dos quais R$ 127.728 são plenamente recuperáveis com automação (taxa de captura de 80%). |
| **Ação Recomendada** | Implementar agente autônomo de triagem/IA conversacional (WhatsApp/site) integrado ao ERP para envio automático de status de entrega e resolução de dúvidas de nível zero. |